In [2]:
from pathlib import Path
from itertools import combinations
from collections import OrderedDict
import re

import cooler
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from matplotlib_venn import venn2
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    matthews_corrcoef,
    roc_curve,
    precision_recall_curve,
)


In [7]:
# ============================================================
# 1. Configuration
# ============================================================

BIN_SIZE = 1_000_000
MCI_ORDER = 3

# 如果候选 node ID 为 1-based，而 node2bin 为 0-based，设置为 -1。
NODE_ID_OFFSET = 0

# 必须使用独立的 pairwise in situ Hi-C。
MCOOL_URI = (
    "/data/xujs/Project/HiPoreC_Promoter_Result/Result/2022-09-27_HiPoreC_to_cool_hic_files/GM12878.Merge.HiPoreC.mcool::resolutions/1000000"
)

NODE2BIN_FILE = "/data/xujs/Project/HiC2PoreC/code/H2P/Analysis_Results/2024-05-31_Model_comparison/GM12878_1Mb_SGMCI_MATCHA_comparison/hg38.1Mb.node2bin.npy"

# 用于估计 MATCHA-style cis/trans Hi-C thresholds。
# 若严格复现 MATCHA 1-Mb 设置，应使用 occurrence count >= 8
# 且不包含 held-out test chromosome 的训练 triplets。
TRAIN_POS_MCI_FILE = "/data/xujs/Project/HiC2PoreC/code/H2P/Analysis_Results/2024-05-31_Model_comparison/MATCHA/1Mb_Temp/train_data.npy"

# Held-out test candidates
TEST_MCI_FILE = "GM12878_1Mb_test_candidates.npy"
TEST_LABEL_FILE = "GM12878_1Mb_test_labels.npy"
TEST_SCORE_FILE = "GM12878_1Mb_test_sgmci_scores.npy"

# Cooler bin table 中的 normalization column。
# MATCHA 使用 KR normalization。
BALANCE_COLUMN = "weight"

# 用于分类性能
SGMCI_CLASS_THRESHOLD = 0.5

# 用于 Venn 图和代表性示例
HIGH_SGMCI_THRESHOLD = 0.8

# 可选：指定 held-out test chromosome。
# 代码将再次确保训练阈值数据不包含这些染色体。
# HELD_OUT_CHROMS = {"chr2"}
HELD_OUT_CHROMS = {}

# Bootstrap confidence intervals
N_BOOTSTRAP = 1000
RANDOM_SEED = 2026

OUTDIR = Path("SGMCI_vs_complete_HiC_clique")
OUTDIR.mkdir(parents=True, exist_ok=True)

In [4]:
# ============================================================
# 2. Candidate-loading functions
# ============================================================

def clean_candidate(row, order=3, offset=0):
    """
    Clean and canonicalize one candidate.

    Negative values are treated as padding.
    MCI nodes are sorted because a triplet is unordered.
    """
    nodes = np.asarray(row, dtype=int).reshape(-1)
    nodes = nodes[nodes >= 0] + offset

    if len(nodes) != order:
        return None

    if len(np.unique(nodes)) != order:
        return None

    return tuple(sorted(nodes.tolist()))


def load_training_mcis(file_name, order=3, offset=0):
    raw = np.load(file_name, allow_pickle=True)

    candidates = []

    for row in raw:
        candidate = clean_candidate(
            row,
            order=order,
            offset=offset,
        )

        if candidate is not None:
            candidates.append(candidate)

    candidates = sorted(set(candidates))

    if not candidates:
        raise ValueError(
            f"No valid training MCIs were found in {file_name}"
        )

    return np.asarray(candidates, dtype=np.int64)


def load_test_bundle(
    mci_file,
    label_file,
    score_file,
    order=3,
    offset=0,
):
    raw_mcis = np.load(mci_file, allow_pickle=True)
    raw_labels = np.load(label_file).reshape(-1)
    raw_scores = np.load(score_file).reshape(-1)

    if not (
        len(raw_mcis)
        == len(raw_labels)
        == len(raw_scores)
    ):
        raise ValueError(
            "The candidate, label, and score arrays have "
            "different lengths."
        )

    records = []

    for row, label, score in zip(
        raw_mcis,
        raw_labels,
        raw_scores,
    ):
        candidate = clean_candidate(
            row,
            order=order,
            offset=offset,
        )

        if candidate is None:
            continue

        if not np.isfinite(label) or label not in (0, 1):
            continue

        if not np.isfinite(score):
            continue

        records.append({
            "candidate": candidate,
            "Label": int(label),
            "SGMCI_score": float(score),
        })

    candidate_df = pd.DataFrame(records)

    if candidate_df.empty:
        raise ValueError(
            "No valid test candidates remained after filtering."
        )

    # Check whether duplicate candidates have conflicting labels.
    label_conflict = (
        candidate_df
        .groupby("candidate")["Label"]
        .nunique()
    )

    conflicting = label_conflict[
        label_conflict > 1
    ]

    if len(conflicting) > 0:
        raise ValueError(
            f"{len(conflicting)} duplicated candidates have "
            "conflicting labels."
        )

    # Retain the highest SGMCI score for duplicated candidates.
    candidate_df = (
        candidate_df
        .sort_values("SGMCI_score", ascending=False)
        .drop_duplicates("candidate", keep="first")
        .reset_index(drop=True)
    )

    mcis = np.asarray(
        candidate_df["candidate"].tolist(),
        dtype=np.int64,
    )

    labels = candidate_df["Label"].to_numpy(dtype=int)
    scores = candidate_df["SGMCI_score"].to_numpy(dtype=float)

    return mcis, labels, scores


train_positive_mcis = load_training_mcis(
    TRAIN_POS_MCI_FILE,
    order=MCI_ORDER,
    offset=NODE_ID_OFFSET,
)

test_mcis, test_labels, test_sgmci_scores = load_test_bundle(
    TEST_MCI_FILE,
    TEST_LABEL_FILE,
    TEST_SCORE_FILE,
    order=MCI_ORDER,
    offset=NODE_ID_OFFSET,
)

print("Training positive MCIs:", len(train_positive_mcis))
print("Test candidates:", len(test_mcis))
print("Test positives:", test_labels.sum())
print("Test negatives:", (test_labels == 0).sum())


Training positive MCIs: 473233
Test candidates: 235623
Test positives: 118272
Test negatives: 117351


In [9]:
# ============================================================
# 3. Load node2bin and align nodes with Cooler bins
# ============================================================

node2bin = np.load(
    NODE2BIN_FILE,
    allow_pickle=True,
).item()

clr = cooler.Cooler(MCOOL_URI)
cool_bins = clr.bins()[:].reset_index().rename(
    columns={"index": "cooler_bin_id"}
)

if BALANCE_COLUMN not in cool_bins.columns:
    raise ValueError(
        f"Normalization column {BALANCE_COLUMN!r} was not found.\n"
        f"Available columns: {cool_bins.columns.tolist()}\n"
        "Use the actual KR/weight column in the .mcool file."
    )


def parse_node_region(region, bin_size=1_000_000):
    """
    Parse:
        chr1:0
        chr1:0-1000000
    """
    match = re.match(
        r"^(chr[^:]+):(\d+)(?:-(\d+))?$",
        str(region),
    )

    if match is None:
        raise ValueError(
            f"Cannot parse node2bin region: {region}"
        )

    chrom = match.group(1)
    start = int(match.group(2))

    end = (
        int(match.group(3))
        if match.group(3) is not None
        else start + bin_size
    )

    return chrom, start, end


cooler_bin_lookup = {
    (str(row.chrom), int(row.start)): int(row.cooler_bin_id)
    for row in cool_bins.itertuples()
}

node_metadata = {}

for node_id, region in node2bin.items():
    chrom, start, end = parse_node_region(
        region,
        bin_size=BIN_SIZE,
    )

    cooler_key = (chrom, start)

    if cooler_key not in cooler_bin_lookup:
        continue

    node_metadata[int(node_id)] = {
        "chrom": chrom,
        "start": start,
        "end": end,
        "region": f"{chrom}:{start}-{end}",
        "cooler_bin_id": cooler_bin_lookup[cooler_key],
    }


required_nodes = np.unique(
    np.concatenate([
        train_positive_mcis.reshape(-1),
        test_mcis.reshape(-1),
    ])
)

missing_nodes = [
    int(node)
    for node in required_nodes
    if int(node) not in node_metadata
]

if missing_nodes:
    raise ValueError(
        f"{len(missing_nodes)} node IDs could not be aligned "
        f"to the Cooler bins. Examples: {missing_nodes[:10]}"
    )


# ============================================================
# 4. Ensure training data exclude held-out chromosomes
# ============================================================

def candidate_chromosomes(candidate):
    return {
        node_metadata[int(node)]["chrom"]
        for node in candidate
    }


if HELD_OUT_CHROMS:
    train_keep_mask = np.asarray([
        len(
            candidate_chromosomes(candidate)
            & HELD_OUT_CHROMS
        ) == 0
        for candidate in train_positive_mcis
    ])

    removed_number = int((~train_keep_mask).sum())

    train_positive_mcis = train_positive_mcis[
        train_keep_mask
    ]

    print(
        "Training MCIs removed because they contained "
        f"held-out chromosomes: {removed_number}"
    )
    print(
        "Training MCIs retained for threshold estimation:",
        len(train_positive_mcis),
    )

# ============================================================
# 5. Load the 1-Mb KR-normalized Hi-C matrix
# ============================================================

hic_matrix = clr.matrix(
    balance=BALANCE_COLUMN,
    sparse=False,
)[:].astype(np.float32)

PAIR_POSITIONS = list(
    combinations(range(MCI_ORDER), 2)
)

N_POSSIBLE_EDGES = len(PAIR_POSITIONS)

print("Hi-C matrix shape:", hic_matrix.shape)
print("Possible pairwise edges per triplet:", N_POSSIBLE_EDGES)


# ============================================================
# 6. Extract pairwise Hi-C contacts
# ============================================================

def extract_pairwise_contacts(mcis):
    """
    Return:
        weights: shape (N, 3)
        cis_mask: shape (N, 3)
    """
    cooler_ids = np.asarray([
        [
            node_metadata[int(node)]["cooler_bin_id"]
            for node in candidate
        ]
        for candidate in mcis
    ], dtype=np.int64)

    chromosomes = np.asarray([
        [
            node_metadata[int(node)]["chrom"]
            for node in candidate
        ]
        for candidate in mcis
    ], dtype=object)

    weights = np.empty(
        (len(mcis), N_POSSIBLE_EDGES),
        dtype=float,
    )

    cis_mask = np.empty(
        (len(mcis), N_POSSIBLE_EDGES),
        dtype=bool,
    )

    for edge_index, (i, j) in enumerate(PAIR_POSITIONS):
        weights[:, edge_index] = hic_matrix[
            cooler_ids[:, i],
            cooler_ids[:, j],
        ]

        cis_mask[:, edge_index] = (
            chromosomes[:, i] == chromosomes[:, j]
        )

    return weights, cis_mask


Hi-C matrix shape: (3103, 3103)
Possible pairwise edges per triplet: 3


In [14]:
# ============================================================
# 7. Derive MATCHA-style cis/trans Hi-C thresholds
# ============================================================

train_weights, train_cis_mask = extract_pairwise_contacts(
    train_positive_mcis
)

# NaN represents bins that could not be normalized and is excluded.
# Real zero contacts are retained.
finite_training_edges = np.isfinite(train_weights)

cis_training_values = train_weights[
    finite_training_edges & train_cis_mask
]

trans_training_values = train_weights[
    finite_training_edges & (~train_cis_mask)
]

if len(cis_training_values) == 0:
    raise ValueError(
        "No finite cis Hi-C contacts were found among "
        "the training-positive triplets."
    )

# cis_threshold = float(np.mean(cis_training_values))
cis_threshold = float(np.percentile(cis_training_values, 50))

trans_threshold = (
    float(np.mean(trans_training_values))
    if len(trans_training_values) > 0
    else np.nan
)

if not np.isfinite(cis_threshold) or cis_threshold <= 0:
    raise ValueError(
        f"Invalid cis threshold: {cis_threshold}. "
        "Check Hi-C normalization and node alignment."
    )

if not np.isfinite(trans_threshold):
    print(
        "Warning: no finite trans threshold was obtained. "
        "Candidates containing trans edges will be excluded."
    )

print(f"Cis threshold:   {cis_threshold:.6g}")
print(f"Trans threshold: {trans_threshold:.6g}")

threshold_df = pd.DataFrame({
    "edge_type": ["cis", "trans"],
    "threshold": [
        cis_threshold,
        trans_threshold,
    ],
    "number_of_training_edges": [
        len(cis_training_values),
        len(trans_training_values),
    ],
})

threshold_df.to_csv(
    OUTDIR / "training_derived_HiC_thresholds.tsv",
    sep="\t",
    index=False,
)

# ============================================================
# 8. Calculate binary complete-clique score
# ============================================================

def calculate_clique_features(
    mcis,
    labels,
    sgmci_scores,
):
    weights, cis_mask = extract_pairwise_contacts(mcis)

    threshold_matrix = np.where(
        cis_mask,
        cis_threshold,
        trans_threshold,
    )

    valid_pairs = (
        np.isfinite(weights)
        & np.isfinite(threshold_matrix)
        & (threshold_matrix > 0)
    )

    # All three pairwise Hi-C values must be available.
    valid_candidates = valid_pairs.all(axis=1)

    supported = (
        valid_pairs
        & (weights >= threshold_matrix)
    )

    supported_number = supported.sum(axis=1)

    # Descriptive only: 0, 1/3, 2/3, or 1.
    supported_edge_fraction = (
        supported_number / N_POSSIBLE_EDGES
    )

    # Main clique baseline:
    # only 3/3 supported edges are assigned 1.
    complete_clique_score = (
        valid_candidates
        & (supported_number == N_POSSIBLE_EDGES)
    ).astype(int)

    records = {
        "candidate": [
            tuple(map(int, candidate))
            for candidate in mcis
        ],
        "Label": labels,
        "SGMCI_score": sgmci_scores,
        "valid_HiC_candidate": valid_candidates,
        "n_supported_edges": supported_number,
        "supported_edge_fraction": supported_edge_fraction,
        "complete_clique_score": complete_clique_score,
    }

    for anchor_index in range(MCI_ORDER):
        records[f"anchor_{anchor_index + 1}"] = [
            node_metadata[int(candidate[anchor_index])]["region"]
            for candidate in mcis
        ]

    for edge_index, (i, j) in enumerate(PAIR_POSITIONS):
        edge_name = f"edge_{i + 1}_{j + 1}"

        records[f"{edge_name}_weight"] = (
            weights[:, edge_index]
        )

        records[f"{edge_name}_threshold"] = (
            threshold_matrix[:, edge_index]
        )

        records[f"{edge_name}_supported"] = (
            supported[:, edge_index]
        )

    candidate_types = []

    for candidate in mcis:
        chroms = candidate_chromosomes(candidate)
        candidate_types.append(
            "cis" if len(chroms) == 1 else "trans"
        )

    records["interaction_type"] = candidate_types

    return pd.DataFrame(records)


test_df = calculate_clique_features(
    test_mcis,
    test_labels,
    test_sgmci_scores,
)

test_df.to_csv(
    OUTDIR / "all_test_candidate_clique_features.tsv",
    sep="\t",
    index=False,
)

In [ ]:
# ============================================================
# 9. Construct the common evaluation dataset
# ============================================================

evaluation_mask = (
    test_df["valid_HiC_candidate"]
    & test_df["Label"].isin([0, 1])
    & np.isfinite(test_df["SGMCI_score"])
)

evaluation_df = test_df.loc[
    evaluation_mask
].copy()

excluded_df = test_df.loc[
    ~evaluation_mask
].copy()

evaluation_df.to_csv(
    OUTDIR / "valid_test_candidates.tsv",
    sep="\t",
    index=False,
)

excluded_df.to_csv(
    OUTDIR / "excluded_invalid_HiC_candidates.tsv",
    sep="\t",
    index=False,
)

print("All test candidates:", len(test_df))
print("Candidates retained:", len(evaluation_df))
print("Candidates excluded:", len(excluded_df))

y_true = evaluation_df["Label"].to_numpy(dtype=int)

if np.unique(y_true).size != 2:
    raise ValueError(
        "Both positive and negative labels are required "
        "to calculate AUROC and AUPRC."
    )


# ============================================================
# 10. Calculate performance
# ============================================================

METHODS = OrderedDict({
    "SGMCI": {
        "score_column": "SGMCI_score",
        "decision_threshold": SGMCI_CLASS_THRESHOLD,
    },
    "Complete Hi-C clique": {
        "score_column": "complete_clique_score",
        "decision_threshold": 0.5,
    },
})


def calculate_performance(
    y,
    scores,
    threshold,
):
    predictions = (
        scores >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y,
        predictions,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "AUROC": roc_auc_score(y, scores),
        "AUPRC": average_precision_score(y, scores),
        "Precision": precision_score(
            y,
            predictions,
            zero_division=0,
        ),
        "Recall": recall_score(
            y,
            predictions,
            zero_division=0,
        ),
        "Specificity": specificity,
        "F1": f1_score(
            y,
            predictions,
            zero_division=0,
        ),
        "MCC": matthews_corrcoef(
            y,
            predictions,
        ),
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn,
    }


performance_records = []

for method, settings in METHODS.items():
    scores = evaluation_df[
        settings["score_column"]
    ].to_numpy(dtype=float)

    result = calculate_performance(
        y_true,
        scores,
        settings["decision_threshold"],
    )

    performance_records.append({
        "Method": method,
        "N": len(y_true),
        "Positive_N": int(y_true.sum()),
        "Negative_N": int((y_true == 0).sum()),
        "Decision_threshold":
            settings["decision_threshold"],
        **result,
    })

performance_df = pd.DataFrame(
    performance_records
)

print(performance_df.to_string(index=False))

performance_df.to_csv(
    OUTDIR / "SGMCI_vs_complete_clique_performance.tsv",
    sep="\t",
    index=False,
)

In [15]:
# ============================================================
# 11. Paired bootstrap confidence intervals
# ============================================================

rng = np.random.default_rng(RANDOM_SEED)

positive_indices = np.where(y_true == 1)[0]
negative_indices = np.where(y_true == 0)[0]

bootstrap_records = []

for bootstrap_id in range(N_BOOTSTRAP):
    sampled_positive = rng.choice(
        positive_indices,
        size=len(positive_indices),
        replace=True,
    )

    sampled_negative = rng.choice(
        negative_indices,
        size=len(negative_indices),
        replace=True,
    )

    sampled_indices = np.concatenate([
        sampled_positive,
        sampled_negative,
    ])

    sampled_y = y_true[sampled_indices]

    for method, settings in METHODS.items():
        full_scores = evaluation_df[
            settings["score_column"]
        ].to_numpy(dtype=float)

        sampled_scores = full_scores[
            sampled_indices
        ]

        sampled_result = calculate_performance(
            sampled_y,
            sampled_scores,
            settings["decision_threshold"],
        )

        for metric in [
            "AUROC",
            "AUPRC",
            "Precision",
            "Recall",
            "F1",
        ]:
            bootstrap_records.append({
                "bootstrap_id": bootstrap_id,
                "Method": method,
                "Metric": metric,
                "Value": sampled_result[metric],
            })

bootstrap_df = pd.DataFrame(
    bootstrap_records
)

bootstrap_df.to_csv(
    OUTDIR / "paired_bootstrap_metrics.tsv",
    sep="\t",
    index=False,
)

bootstrap_ci = (
    bootstrap_df
    .groupby(["Method", "Metric"])["Value"]
    .agg(
        CI_lower=lambda x: np.quantile(x, 0.025),
        CI_upper=lambda x: np.quantile(x, 0.975),
    )
    .reset_index()
)

bootstrap_ci.to_csv(
    OUTDIR / "bootstrap_95CI.tsv",
    sep="\t",
    index=False,
)


# ============================================================
# 12. Visualization 1: performance bar plot
# ============================================================

sns.set_theme(style="white", context="notebook")

method_colors = {
    "SGMCI": "#69B3A2",
    "Complete Hi-C clique": "#E99675",
}

plot_metrics = [
    "AUROC",
    "AUPRC",
    "Precision",
    "Recall",
    "F1",
]

metric_long = performance_df.melt(
    id_vars="Method",
    value_vars=plot_metrics,
    var_name="Metric",
    value_name="Value",
)

fig, ax = plt.subplots(
    figsize=(10, 5.5),
    dpi=180,
)

sns.barplot(
    data=metric_long,
    x="Metric",
    y="Value",
    hue="Method",
    palette=method_colors,
    errorbar=None,
    ax=ax,
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.3f",
        padding=2,
        fontsize=8,
    )

ax.set_ylim(0, 1.07)
ax.set_xlabel("")
ax.set_ylabel("Performance")
ax.set_title(
    "SGMCI versus complete Hi-C clique"
)

ax.legend(
    title="",
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)

sns.despine(ax=ax)
plt.tight_layout()

plt.savefig(
    OUTDIR / "performance_barplot.pdf",
    bbox_inches="tight",
)

plt.savefig(
    OUTDIR / "performance_barplot.png",
    dpi=300,
    bbox_inches="tight",
)

plt.close()


# ============================================================
# 13. Visualization 2: ROC and PR curves
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11, 4.8),
    dpi=180,
)

for method, settings in METHODS.items():
    scores = evaluation_df[
        settings["score_column"]
    ].to_numpy(dtype=float)

    fpr, tpr, _ = roc_curve(
        y_true,
        scores,
        drop_intermediate=False,
    )

    precision, recall, _ = precision_recall_curve(
        y_true,
        scores,
    )

    auroc = roc_auc_score(
        y_true,
        scores,
    )

    auprc = average_precision_score(
        y_true,
        scores,
    )

    drawstyle = (
        "steps-post"
        if method == "Complete Hi-C clique"
        else "default"
    )

    axes[0].plot(
        fpr,
        tpr,
        color=method_colors[method],
        linewidth=2,
        drawstyle=drawstyle,
        label=f"{method}: {auroc:.3f}",
    )

    axes[1].plot(
        recall,
        precision,
        color=method_colors[method],
        linewidth=2,
        drawstyle=drawstyle,
        label=f"{method}: {auprc:.3f}",
    )

axes[0].plot(
    [0, 1],
    [0, 1],
    "--",
    color="grey",
    linewidth=1,
)

axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].set_title("ROC curves")

prevalence = y_true.mean()

axes[1].axhline(
    prevalence,
    linestyle="--",
    color="grey",
    linewidth=1,
    label=f"No-skill baseline: {prevalence:.3f}",
)

axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-recall curves")

for ax in axes:
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.legend(frameon=False, fontsize=8)
    sns.despine(ax=ax)

plt.tight_layout()

plt.savefig(
    OUTDIR / "ROC_and_PR_curves.pdf",
    bbox_inches="tight",
)

plt.savefig(
    OUTDIR / "ROC_and_PR_curves.png",
    dpi=300,
    bbox_inches="tight",
)

plt.close()


# ============================================================
# 14. Visualization 3: Venn diagram
# ============================================================

sgmci_set = set(
    evaluation_df.loc[
        evaluation_df["SGMCI_score"]
        >= HIGH_SGMCI_THRESHOLD,
        "candidate",
    ]
)

clique_set = set(
    evaluation_df.loc[
        evaluation_df["complete_clique_score"] == 1,
        "candidate",
    ]
)

shared_set = sgmci_set & clique_set
sgmci_only_set = sgmci_set - clique_set
clique_only_set = clique_set - sgmci_set

n_sgmci = len(sgmci_set)
n_clique = len(clique_set)
n_shared = len(shared_set)

clique_recovery = (
    n_shared / n_clique
    if n_clique > 0
    else np.nan
)

sgmci_clique_fraction = (
    n_shared / n_sgmci
    if n_sgmci > 0
    else np.nan
)

print("\nCandidate-set overlap")
print("High-score SGMCI candidates:", n_sgmci)
print("Complete Hi-C cliques:", n_clique)
print("Shared:", n_shared)
print("SGMCI high-score non-cliques:", len(sgmci_only_set))
print("Clique-only:", len(clique_only_set))
print("Fraction of cliques recovered:", clique_recovery)
print(
    "Fraction of SGMCI candidates that are cliques:",
    sgmci_clique_fraction,
)

overlap_df = pd.DataFrame({
    "Category": [
        "High-score SGMCI",
        "Complete Hi-C clique",
        "Shared",
        "SGMCI-only",
        "Clique-only",
        "Clique recovery fraction",
        "SGMCI complete-clique fraction",
    ],
    "Value": [
        n_sgmci,
        n_clique,
        n_shared,
        len(sgmci_only_set),
        len(clique_only_set),
        clique_recovery,
        sgmci_clique_fraction,
    ],
})

overlap_df.to_csv(
    OUTDIR / "candidate_set_overlap.tsv",
    sep="\t",
    index=False,
)

fig, ax = plt.subplots(
    figsize=(7, 6),
    dpi=180,
)

venn2(
    subsets=(
        len(sgmci_only_set),
        len(clique_only_set),
        len(shared_set),
    ),
    set_labels=(
        f"SGMCI score >= {HIGH_SGMCI_THRESHOLD}",
        "Complete Hi-C clique",
    ),
    set_colors=(
        "#DFA0A5",
        "#E7DFC8",
    ),
    alpha=0.8,
    ax=ax,
)

ax.set_title(
    "SGMCI predictions versus complete Hi-C cliques"
)

plt.tight_layout()

plt.savefig(
    OUTDIR / "candidate_set_venn.pdf",
    bbox_inches="tight",
)

plt.savefig(
    OUTDIR / "candidate_set_venn.png",
    dpi=300,
    bbox_inches="tight",
)

plt.close()


# ============================================================
# 15. Visualization 4: Hi-C edge support by SGMCI decile
# ============================================================

decile_labels = [
    f"D{i}"
    for i in range(1, 11)
]

evaluation_df["SGMCI_score_decile"] = pd.qcut(
    evaluation_df["SGMCI_score"].rank(method="first"),
    q=10,
    labels=decile_labels,
)

edge_fraction = pd.crosstab(
    evaluation_df["SGMCI_score_decile"],
    evaluation_df["n_supported_edges"],
    normalize="index",
).reindex(
    index=decile_labels,
    columns=[0, 1, 2, 3],
    fill_value=0,
)

complete_fraction = (
    evaluation_df
    .groupby(
        "SGMCI_score_decile",
        observed=False,
    )["complete_clique_score"]
    .mean()
    .reindex(decile_labels)
)

edge_fraction.to_csv(
    OUTDIR / "edge_support_by_SGMCI_decile.tsv",
    sep="\t",
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5),
    dpi=180,
)

edge_fraction.plot(
    kind="bar",
    stacked=True,
    color=[
        "#EEEEEE",
        "#BDD7E7",
        "#6BAED6",
        "#2171B5",
    ],
    width=0.82,
    ax=axes[0],
)

axes[0].set_xlabel(
    "SGMCI prediction-score decile"
)

axes[0].set_ylabel(
    "Fraction of candidate triplets"
)

axes[0].set_title(
    "Number of supported pairwise Hi-C edges"
)

axes[0].tick_params(
    axis="x",
    rotation=0,
)

axes[0].legend(
    title="Supported edges",
    frameon=False,
)

axes[1].plot(
    decile_labels,
    complete_fraction.to_numpy(),
    marker="o",
    linewidth=2,
    color="#D7301F",
)

axes[1].set_ylim(0, 1)
axes[1].set_xlabel(
    "SGMCI prediction-score decile"
)

axes[1].set_ylabel(
    "Fraction of complete Hi-C cliques"
)

axes[1].set_title(
    "Complete-clique frequency"
)

for ax in axes:
    sns.despine(ax=ax)

plt.tight_layout()

plt.savefig(
    OUTDIR / "HiC_support_by_SGMCI_decile.pdf",
    bbox_inches="tight",
)

plt.savefig(
    OUTDIR / "HiC_support_by_SGMCI_decile.png",
    dpi=300,
    bbox_inches="tight",
)

plt.close()


# ============================================================
# 16. Select and export representative non-clique candidate
# ============================================================

nonclique_df = evaluation_df[
    (evaluation_df["SGMCI_score"] >= HIGH_SGMCI_THRESHOLD)
    & (evaluation_df["complete_clique_score"] == 0)
    & (evaluation_df["interaction_type"] == "cis")
].copy()

# Preferred example:
# held-out HiPore-C positive + 2 supported Hi-C edges
nonclique_df["selection_priority"] = np.select(
    [
        (
            (nonclique_df["Label"] == 1)
            & (nonclique_df["n_supported_edges"] == 2)
        ),
        nonclique_df["Label"] == 1,
        nonclique_df["n_supported_edges"] == 2,
    ],
    [1, 2, 3],
    default=4,
)

nonclique_df = nonclique_df.sort_values(
    [
        "selection_priority",
        "SGMCI_score",
        "n_supported_edges",
    ],
    ascending=[
        True,
        False,
        False,
    ],
)

nonclique_df.to_csv(
    OUTDIR / "high_score_non_complete_clique_candidates.tsv",
    sep="\t",
    index=False,
)

print(
    "\nHigh-score cis non-complete-clique candidates:",
    len(nonclique_df),
)

if len(nonclique_df) > 0:
    representative = nonclique_df.iloc[0]

    representative.to_frame().T.to_csv(
        OUTDIR / "representative_nonclique_candidate.tsv",
        sep="\t",
        index=False,
    )

    bed_rows = []

    for anchor_index in range(1, MCI_ORDER + 1):
        region = representative[
            f"anchor_{anchor_index}"
        ]

        chrom, start, end = parse_node_region(
            region,
            bin_size=BIN_SIZE,
        )

        bed_rows.append({
            "chrom": chrom,
            "start": start,
            "end": end,
            "name": (
                f"anchor_{anchor_index}|"
                f"SGMCI={representative['SGMCI_score']:.4f}|"
                f"edges={representative['n_supported_edges']}/3"
            ),
        })

    pd.DataFrame(bed_rows).to_csv(
        OUTDIR / "representative_nonclique_candidate.bed",
        sep="\t",
        header=False,
        index=False,
    )

    print("\nRepresentative candidate:")
    print(
        representative[
            [
                "anchor_1",
                "anchor_2",
                "anchor_3",
                "Label",
                "SGMCI_score",
                "n_supported_edges",
                "supported_edge_fraction",
            ]
        ]
    )


All test candidates: 235623
Candidates retained: 217700
Candidates excluded: 17923
              Method      N  Positive_N  Negative_N  Decision_threshold    AUROC    AUPRC  Precision   Recall  Specificity       F1      MCC     TP    FP     TN    FN
               SGMCI 217700      114838      102862                 0.5 0.842089 0.838347   0.629161 0.979824     0.355233 0.766280 0.436836 112521 66322  36540  2317
Complete Hi-C clique 217700      114838      102862                 0.5 0.564384 0.587536   0.986660 0.130741     0.998026 0.230887 0.252125  15014   203 102659 99824

Candidate-set overlap
High-score SGMCI candidates: 144345
Complete Hi-C cliques: 15217
Shared: 15076
SGMCI high-score non-cliques: 129269
Clique-only: 141
Fraction of cliques recovered: 0.9907340474469344
Fraction of SGMCI candidates that are cliques: 0.10444421351622848

High-score cis non-complete-clique candidates: 105505

Representative candidate:
anchor_1                   chr13:25000000-26000000
anchor_2  

In [ ]:

# ============================================================
# 17. Visualization 5: local Hi-C heatmap and triplet graph
# ============================================================

def plot_representative_candidate(
    candidate_row,
    cooler_object,
    balance_column,
    output_file,
    flank_bins=3,
):
    candidate = tuple(
        map(int, candidate_row["candidate"])
    )

    metadata = [
        node_metadata[node]
        for node in candidate
    ]

    chromosomes = {
        item["chrom"]
        for item in metadata
    }

    if len(chromosomes) != 1:
        raise ValueError(
            "Local heatmap visualization requires a cis candidate."
        )

    chrom = metadata[0]["chrom"]

    anchor_starts = [
        item["start"]
        for item in metadata
    ]

    anchor_ends = [
        item["end"]
        for item in metadata
    ]

    chromosome_size = int(
        cooler_object.chromsizes[chrom]
    )

    window_start = max(
        0,
        min(anchor_starts) - flank_bins * BIN_SIZE,
    )

    window_end = min(
        chromosome_size,
        max(anchor_ends) + flank_bins * BIN_SIZE,
    )

    region = (
        f"{chrom}:{window_start}-{window_end}"
    )

    local_matrix = cooler_object.matrix(
        balance=balance_column,
        sparse=False,
    ).fetch(region)

    finite_values = local_matrix[
        np.isfinite(local_matrix)
        & (local_matrix > 0)
    ]

    vmax = (
        np.quantile(finite_values, 0.99)
        if len(finite_values) > 0
        else 1
    )

    plot_matrix = np.nan_to_num(
        local_matrix,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12, 5),
        dpi=180,
        gridspec_kw={
            "width_ratios": [1.5, 1]
        },
    )

    extent = [
        window_start / 1e6,
        window_end / 1e6,
        window_start / 1e6,
        window_end / 1e6,
    ]

    image = axes[0].imshow(
        plot_matrix,
        origin="lower",
        cmap="Reds",
        vmin=0,
        vmax=vmax,
        extent=extent,
        aspect="equal",
    )

    anchor_centers_mb = [
        (item["start"] + item["end"])
        / 2e6
        for item in metadata
    ]

    anchor_colors = [
        "#1B9E77",
        "#7570B3",
        "#D95F02",
    ]

    for index, center in enumerate(
        anchor_centers_mb
    ):
        axes[0].axvline(
            center,
            color=anchor_colors[index],
            linewidth=1.5,
            linestyle="--",
        )

        axes[0].axhline(
            center,
            color=anchor_colors[index],
            linewidth=1.5,
            linestyle="--",
        )

    axes[0].set_xlabel(
        f"{chrom} position (Mb)"
    )

    axes[0].set_ylabel(
        f"{chrom} position (Mb)"
    )

    axes[0].set_title(
        "Local KR-normalized Hi-C map"
    )

    plt.colorbar(
        image,
        ax=axes[0],
        fraction=0.046,
        pad=0.04,
        label="Normalized Hi-C contact",
    )

    # Triplet topology
    node_positions = {
        0: (0.5, 0.88),
        1: (0.15, 0.18),
        2: (0.85, 0.18),
    }

    for edge_index, (i, j) in enumerate(
        PAIR_POSITIONS
    ):
        supported = bool(
            candidate_row[
                f"edge_{i + 1}_{j + 1}_supported"
            ]
        )

        x1, y1 = node_positions[i]
        x2, y2 = node_positions[j]

        axes[1].plot(
            [x1, x2],
            [y1, y2],
            color=(
                "#2CA25F"
                if supported
                else "#CB181D"
            ),
            linewidth=3,
            linestyle=(
                "-"
                if supported
                else "--"
            ),
            alpha=0.9,
        )

    for index, item in enumerate(metadata):
        x, y = node_positions[index]

        axes[1].scatter(
            x,
            y,
            s=800,
            color=anchor_colors[index],
            edgecolor="black",
            linewidth=1.5,
            zorder=3,
        )

        axes[1].text(
            x,
            y,
            f"A{index + 1}",
            ha="center",
            va="center",
            color="white",
            fontsize=12,
            fontweight="bold",
            zorder=4,
        )

        axes[1].text(
            x,
            y - 0.13,
            item["region"],
            ha="center",
            va="top",
            fontsize=8,
        )

    axes[1].text(
        0.5,
        0.02,
        (
            f"SGMCI score = "
            f"{candidate_row['SGMCI_score']:.3f}\n"
            f"HiPore-C label = "
            f"{int(candidate_row['Label'])}\n"
            f"Supported edges = "
            f"{int(candidate_row['n_supported_edges'])}/3"
        ),
        ha="center",
        va="bottom",
        fontsize=10,
    )

    axes[1].set_xlim(-0.05, 1.05)
    axes[1].set_ylim(-0.05, 1.05)
    axes[1].axis("off")
    axes[1].set_title(
        "Pairwise support of the candidate triplet"
    )

    fig.suptitle(
        "Representative high-scoring non-complete-clique candidate",
        fontsize=14,
    )

    plt.tight_layout()

    plt.savefig(
        output_file,
        bbox_inches="tight",
    )

    plt.close()


if len(nonclique_df) > 0:
    plot_representative_candidate(
        candidate_row=representative,
        cooler_object=clr,
        balance_column=BALANCE_COLUMN,
        output_file=(
            OUTDIR
            / "representative_nonclique_HiC_heatmap.pdf"
        ),
    )


print(f"\nAnalysis completed: {OUTDIR.resolve()}")